<a href="https://colab.research.google.com/github/dequiroz/1MTR53_RobIA/blob/main/03_Cinem%C3%A1tica_Robots_moviles/01_Marcos_Referencia_y_Pose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://drive.google.com/uc?export=view&id=1FsTa2YzWwxY-HBkd2EOXAlkPJ1zpikWv" alt="drawing" width="150"/>


<p align="center">
<img src="https://drive.google.com/uc?export=view&id=1VBn7nKlruxCPLHH_UuD8B2pkvKP7JLRd" alt="drawing" width="800"/>


</br>

<img src="https://drive.google.com/uc?export=view&id=1tpg1CJh4or-VMrY-Qip25hgedgwy1Y8-" alt="drawing" width="800"/>
</p>


# Marcos de Referencia, Postura y Transformaciones Geométricas
* **Responsable del material:** Diego Quiroz Velasquez
* Pontificia Universidad Católica del Perú
* Sección Ingeniería Mecatrónica

---
> **Descripción:**
> Este cuaderno interactivo constituye un material complementario para el curso de Robótica e Inteligencia Artificial, alineado con la **Unidad 3: Cinemática de Robots Móviles**. Su propósito es trasladar los conceptos teóricos de álgebra lineal, matrices de rotación y marcos de referencia (sistemas inerciales vs. sistemas locales) hacia una implementación práctica utilizando Python y la librería `numpy`.

<details>
<summary><b>Historial de Versiones del Documento (Hacer clic para desplegar)</b></summary>

<br>

| Versión | Fecha | Autor / Rol | Descripción del Cambio |
| :---: | :---: | :--- | :--- |
| **v1.0** | 29/07/2026 | Diego Quiroz | Estructuración inicial de cinemática y transformaciones geométricas. |

---
</details>
<br>

<img src="https://drive.google.com/uc?export=view&id=1vgFoFHEKMN0WcAFlKS2hPMg2KuBm3sp9" alt="drawing"/>



## 1. Introducción: Postura (Pose) en Robótica Móvil

Como se revisó en la **Clase 1 y 2**, existe una diferencia fundamental entre un brazo manipulador y un robot móvil. Mientras el manipulador tiene una base fija, el robot móvil opera desplazando su base con respecto a un sistema de coordenadas global o inercial (fijo).

Para representar matemáticamente dónde está un robot terrestre con ruedas en un entorno 2D, necesitamos 3 grados de libertad:
* **Posición:** Coordenadas cartesianas $(x, y)$.
* **Orientación:** Ángulo de giro respecto al eje inercial $X$, denotado usualmente como $\varphi$ o $\theta$.

El vector de configuración o **postura (pose)** se escribe como:

$$ \xi = \begin{bmatrix} x \\ y \\ \theta \end{bmatrix} $$

Para realizar cálculos algebraicos y matriciales en Python de manera eficiente, utilizaremos la librería `numpy`.

In [1]:
# ==============================================================================
# 1. REPRESENTACIÓN DE LA POSTURA MEDIANTE ARREGLOS (NUMPY)
# ==============================================================================
import numpy as np

# Definimos las coordenadas del robot en el sistema global {I}
x_robot = 5.2    # metros
y_robot = 3.8    # metros
theta_deg = 45.0 # grados

# Es fundamental trabajar los ángulos en radianes en cinemática
theta_rad = np.deg2rad(theta_deg)

# Representación del vector de postura (pose) como un arreglo de numpy
postura_robot = np.array([
    [x_robot],
    [y_robot],
    [theta_rad]
])

print("--- Vector de Configuración del Robot (Pose) ---")
print(f"X: {postura_robot[0][0]:.2f} m")
print(f"Y: {postura_robot[1][0]:.2f} m")
print(f"Theta: {postura_robot[2][0]:.4f} rad ({theta_deg} °)")

--- Vector de Configuración del Robot (Pose) ---
X: 5.20 m
Y: 3.80 m
Theta: 0.7854 rad (45.0 °)


## 2. Matrices de Rotación (Representación de la Orientación)

Para relacionar mediciones hechas por los sensores a bordo del robot (sistema de referencia local $\{R\}$) con el mundo real (sistema de referencia inercial $\{I\}$), necesitamos **Transformaciones Geométricas**.

Como vimos en la teoría, la matriz de rotación básica en un plano 2D (o rotación alrededor del eje Z en 3D) permite alinear la orientación. La matriz de rotación $R_R^I$ (del robot al inercial) se define como:

$$ R_R^I = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} $$

Esta matriz nos permite tomar un vector de velocidad o un punto descrito en el marco del robot y "girarlo" para entender su dirección en el marco global.

In [2]:
# ==============================================================================
# 2. CREACIÓN DE MATRICES DE ROTACIÓN EN PYTHON
# ==============================================================================

def generar_matriz_rotacion_2d(angulo_rad):
    """
    Genera una matriz de rotación 2D dado un ángulo en radianes.
    """
    c = np.cos(angulo_rad)
    s = np.sin(angulo_rad)

    # Declaración de la matriz usando numpy
    R = np.array([
        [c, -s],
        [s,  c]
    ])
    return R

# Obtenemos la matriz de rotación para la postura actual de nuestro robot (45 grados)
matriz_rotacion = generar_matriz_rotacion_2d(postura_robot[2][0])

print("--- Matriz de Rotación (Robot -> Global) ---")
print(np.round(matriz_rotacion, decimals=3))

--- Matriz de Rotación (Robot -> Global) ---
[[ 0.707 -0.707]
 [ 0.707  0.707]]


## 3. Transformación Rotacional y Traslacional Completa

Imaginemos que el robot tiene un sensor LiDAR montado exactamente en su centro. Este sensor detecta un obstáculo a unas coordenadas $(x_o^R, y_o^R)$ **desde la perspectiva del robot** (es decir, en el sistema local $\{R\}$).

Para que el robot sepa dónde está el obstáculo en el **mapa global** $\{I\}$, debemos aplicar la rotación y luego sumar la traslación (la posición actual del robot):

$$ P^I = R_R^I \cdot P^R + P_{robot}^I $$

Donde:
* $P^I$: Posición del obstáculo en el marco Inercial/Global.
* $P^R$: Posición del obstáculo medido por el sensor en el marco del Robot.
* $P_{robot}^I$: Posición $(x, y)$ actual del robot en el marco Global.

In [3]:
# ==============================================================================
# 3. TRANSFORMACIÓN DE COORDENADAS (MAPEO DE SENSORES)
# ==============================================================================

# 1. El sensor detecta un obstáculo a 2 metros adelante (eje x local) y 0 metros al lado
P_R = np.array([
    [2.0],
    [0.0]
])

print(f"Obstáculo detectado en coordenadas locales (robot): \n{P_R}\n")

# 2. Extraemos el vector de traslación del robot (solo X e Y)
# Usamos slicing para tomar las primeras dos filas
P_robot_I = postura_robot[0:2]

# 3. Aplicamos la transformación lineal
# En numpy, el símbolo '@' o la función np.dot() se utilizan para multiplicación matricial
P_I = (matriz_rotacion @ P_R) + P_robot_I

print("--- Transformación Geométrica Completada ---")
print(f"Coordenadas del obstáculo en el MAPA GLOBAL:")
print(f"X global: {P_I[0][0]:.2f} m")
print(f"Y global: {P_I[1][0]:.2f} m")

Obstáculo detectado en coordenadas locales (robot): 
[[2.]
 [0.]]

--- Transformación Geométrica Completada ---
Coordenadas del obstáculo en el MAPA GLOBAL:
X global: 6.61 m
Y global: 5.21 m
